# Label Encoding vs. One-Hot Encoding for Predictive Maintenance

This notebook demonstrates two common ways to encode categorical maintenance data for machine-learning workflows:

- **Label/Ordinal Encoding:** useful when categories have a meaningful order.
- **One-Hot Encoding:** useful when categories are nominal and have no natural order.

### Predictive-maintenance examples
- **Condition:** Good → Fair → Poor → Critical → ordered categories.
- **Failure Mode:** Mechanical, Electrical, Hydraulic → unordered categories.

> **Important:** In scikit-learn, `LabelEncoder` is primarily intended for encoding a target variable (`y`). For an ordered input feature (`X`), `OrdinalEncoder` is generally more appropriate. This notebook demonstrates both concepts clearly.

## 1. Import the required libraries

In [1]:
import pandas as pd

# LabelEncoder: typically used for a target variable (y)
# OrdinalEncoder: appropriate for ordered categorical input features (X)
# OneHotEncoder: appropriate for nominal categorical input features (X)
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder

## 2. Create a predictive-maintenance dataset

Each row represents a machine observation. We include two categorical variables:

- `Condition`: has a natural order.
- `Failure_Mode`: has no natural order.

In [2]:
data = {
    "Machine_ID": ["M-101", "M-102", "M-103", "M-104", "M-105", "M-106"],
    "Condition": ["Good", "Poor", "Fair", "Critical", "Good", "Poor"],
    "Failure_Mode": [
        "Mechanical",
        "Electrical",
        "Hydraulic",
        "Mechanical",
        "Electrical",
        "Hydraulic"
    ]
}

df = pd.DataFrame(data)
df

,Machine_ID,Condition,Failure_Mode
0,M-101,Good,Mechanical
1,M-102,Poor,Electrical
2,M-103,Fair,Hydraulic
3,M-104,Critical,Mechanical
4,M-105,Good,Electrical
5,M-106,Poor,Hydraulic


## 3. Label encoding a target variable

Suppose our predictive-maintenance task is to predict the **failure mode** of a machine. Here, `Failure_Mode` is the target (`y`).

`LabelEncoder` converts each target category into an integer label.

In [3]:
# Create a LabelEncoder for the target variable
label_encoder = LabelEncoder()

# Encode the categorical target
df["Failure_Mode_Encoded"] = label_encoder.fit_transform(df["Failure_Mode"])

df[["Failure_Mode", "Failure_Mode_Encoded"]]

,Failure_Mode,Failure_Mode_Encoded
0,Mechanical,2
1,Electrical,0
2,Hydraulic,1
3,Mechanical,2
4,Electrical,0
5,Hydraulic,1


### View the target-label mapping

The integer values are labels for the target classes. They do not mean that one failure mode is numerically larger or more severe than another.

In [4]:
target_mapping = pd.DataFrame({
    "Failure_Mode": label_encoder.classes_,
    "Encoded_Label": label_encoder.transform(label_encoder.classes_)
})

target_mapping

,Failure_Mode,Encoded_Label
0,Electrical,0
1,Hydraulic,1
2,Mechanical,2


## 4. Ordinal encoding an ordered maintenance feature

`Condition` has a meaningful order:

**Good < Fair < Poor < Critical**

Because the order carries information, we can represent the categories with ordered numerical codes.

In [5]:
# Explicitly define the engineering order of the condition categories.
# This prevents the encoding from being based on alphabetical order.
condition_order = [["Good", "Fair", "Poor", "Critical"]]

ordinal_encoder = OrdinalEncoder(categories=condition_order)

df["Condition_Encoded"] = ordinal_encoder.fit_transform(
    df[["Condition"]]
).astype(int)

df[["Condition", "Condition_Encoded"]]

,Condition,Condition_Encoded
0,Good,0
1,Poor,2
2,Fair,1
3,Critical,3
4,Good,0
5,Poor,2


## 5. One-hot encoding a nominal maintenance feature

`Failure_Mode` is **nominal**: Mechanical, Electrical, and Hydraulic do not have a natural ranking.

One-hot encoding creates one binary column for each category.

In [6]:
# Create a OneHotEncoder for the nominal categorical feature.
# sparse_output=False makes the result easier to inspect in a notebook.
onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

failure_mode_onehot = onehot_encoder.fit_transform(
    df[["Failure_Mode"]]
)

# Get readable column names such as Failure_Mode_Electrical.
onehot_columns = onehot_encoder.get_feature_names_out(["Failure_Mode"])

onehot_df = pd.DataFrame(
    failure_mode_onehot.astype(int),
    columns=onehot_columns,
    index=df.index
)

onehot_df

,Failure_Mode_Electrical,Failure_Mode_Hydraulic,Failure_Mode_Mechanical
0,0,0,1
1,1,0,0
2,0,1,0
3,0,0,1
4,1,0,0
5,0,1,0


## 6. Combine the encoded features

We can combine the original maintenance table with the encoded representations to see the complete result.

In [7]:
# Combine the original data with the two encoding examples.
encoded_df = pd.concat(
    [
        df[["Machine_ID", "Condition", "Condition_Encoded", "Failure_Mode"]],
        onehot_df
    ],
    axis=1
)

encoded_df

,Machine_ID,Condition,Condition_Encoded,Failure_Mode,Failure_Mode_Electrical,Failure_Mode_Hydraulic,Failure_Mode_Mechanical
0,M-101,Good,0,Mechanical,0,0,1
1,M-102,Poor,2,Electrical,1,0,0
2,M-103,Fair,1,Hydraulic,0,1,0
3,M-104,Critical,3,Mechanical,0,0,1
4,M-105,Good,0,Electrical,1,0,0
5,M-106,Poor,2,Hydraulic,0,1,0


## 7. Compare the two approaches

| Encoding | When to use | Maintenance example |
|---|---|---|
| **LabelEncoder** | Mainly for a categorical **target (`y`)** | Predict failure mode: Mechanical / Electrical / Hydraulic |
| **OrdinalEncoder** | Ordered categorical **input (`X`)** | Condition: Good → Fair → Poor → Critical |
| **OneHotEncoder** | Unordered categorical **input (`X`)** | Failure mode, asset type, manufacturer |

### Simple rule
**Ordered categories → Ordinal Encoding**  
**Unordered categories → One-Hot Encoding**  
**Categorical target → Label Encoding can be used**

## 8. A common predictive-maintenance mistake

Suppose we encode failure modes as:

- Electrical = 0
- Hydraulic = 1
- Mechanical = 2

It would be incorrect to interpret **Mechanical = 2** as being "greater" or "more severe" than **Electrical = 0**. These numbers are simply class labels.

For a nominal input feature, one-hot encoding avoids introducing a false numerical ordering.

## 9. Quick ClassPoint-style challenge

**A predictive-maintenance dataset contains `Manufacturer = {Siemens, ABB, GE}` and `Condition = {Good, Fair, Poor, Critical}`. Which encoding choice is most appropriate?**

Think first, then check the principle:

- `Manufacturer` → **One-Hot Encoding** (nominal)
- `Condition` → **Ordinal Encoding** (ordered)

The key question is not whether the feature contains text. The key question is whether the categories have a **meaningful order**.